In [1]:
%load_ext autoreload
%autoreload 2

import tiktoken
import torch
from utils import plot_metrics,  tokenize, group
import os

num_cores = os.cpu_count()
enc = tiktoken.get_encoding("gpt2")
enc.n_vocab

50257

In [ ]:
from model import ModelConfig

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device} with {num_cores} cores")

config = ModelConfig()

In [ ]:
import datasets

ds = datasets.load_dataset('roneneldan/TinyStories', streaming=True)
demo_samples = ds['train'].take(3)

for i, example in enumerate(demo_samples):
    print(f"--- Story {i+1} ---")
    print(example['text'])
    

In [ ]:
dataset = ds.map(
    tokenize, 
    batched=True, 
    remove_columns=["text"]
).map(
    group, 
    batched=True, 
    fn_kwargs={"block_size": config.context_length}, 
).with_format('torch')

example = next(iter(dataset['train']))
example['encoded'].shape, enc.decode(example['encoded'].numpy())

In [ ]:
from torch.utils.data import DataLoader

train_dataset = dataset["train"].shuffle(buffer_size=10000)
val_dataset = dataset["validation"]

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, pin_memory=True, num_workers=min(4, num_cores))  # type: ignore
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, pin_memory=True, num_workers=min(2, num_cores))  # type: ignore

In [ ]:
from torch import optim, nn
import torch.nn.functional as F
import torchmetrics
from model import BigramLanguageModel, Sampler, Transformer

model = Transformer(config).to(device)
optimizer = optim.AdamW(model.parameters(), config.learning_rate)
metrics = torchmetrics.MetricCollection({
    'loss': torchmetrics.MeanMetric()
})

def compute_loss(logits: torch.Tensor, targets: torch.Tensor):
    B, T, C = logits.shape
    return F.cross_entropy(logits.reshape(B * T, C), targets.reshape(B * T))

def train_step(model: nn.Module, batch, optimizer: optim.Optimizer, metrics: torchmetrics.MetricCollection):
    data = batch["encoded"].to(device)
    x = data[:, :-1]
    y = data[:, 1:]

    optimizer.zero_grad(set_to_none=True)

    logits = model(x)  
    loss = compute_loss(logits, y)

    loss.backward()
    optimizer.step()
    metrics.update(value=loss.item())

@torch.no_grad()
def eval_step(model: nn.Module, batch, metrics: torchmetrics.MetricCollection):
    data = batch["encoded"].to(device)
    x = data[:, :-1]
    y = data[:, 1:]
    logits = model(x)
    loss = compute_loss(logits, y)
    metrics.update(value=loss.item())


In [ ]:
x, y = example['encoded'][:-1].to(device), example['encoded'][1:].to(device)
logits = model(x.unsqueeze(0))
loss = compute_loss(logits, y.unsqueeze(0))
print(f"Initial loss: {loss.item():.4f}")


# Loss is expected to be log(n_vocab)

sampler = Sampler(model)
context = torch.tensor([enc.encode("Once upon a time")], device=device)
generated = sampler.sample(context, max_length=50)

enc.decode(generated[0].cpu().numpy().tolist())

In [ ]:
for step, batch in enumerate(train_loader):
    train_step(model, batch, optimizer, metrics)
    break
    if step % 50 == 0:
        print(f"Step {step}: {metrics.compute()['loss']}")
    if step > 1000:
        break

print(metrics.compute())

In [ ]:
metrics_history = {
    'train_loss': [],
    'val_loss': [],
}

for epoch in range(1):
    metrics.reset()
    model.train()
    for step, batch in enumerate(train_loader):
        train_step(model, batch, optimizer, metrics)
        if (step + 1) % 1000 == 0:
            print(f"Training loss: {metrics.compute()['loss']} at step = {step + 1}")
    
    for metric, value in metrics.compute().items():
        metrics_history[f"train_{metric.lower()}"].append(value.item())

    # --- Evaluation ---
    metrics.reset()
    model.eval()
    for batch in val_loader:
        eval_step(model, batch, metrics)
    
    for metric, value in metrics.compute().items():
        metrics_history[f"test_{metric.lower()}"].append(value.item())
    plot_metrics(metrics_history)
    
    latest_acc = metrics_history['test_accuracy'][-1]
    latest_loss = metrics_history['test_loss'][-1]
    print(f"Epoch {epoch+1} complete. Test Acc: {latest_acc:.4f}, Test Loss: {latest_loss:.4f}")

# Attention

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from model import AttentionHead
import torch

# x = torch.randint(1, 10,  (1, 4, 2), dtype=torch.float16)
x = torch.randn((1, 4 ,2))
attn = AttentionHead(CONTEXT_LENGTH, 2, 2)

scores = attn(x)
x, scores

In [ ]:
# self attention
B, T, C = 4, 8, 32

x = torch.randn(B, T, C)
mask = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(mask == 0, float('-inf'))
wei = F.softmax(wei, -1)

out = wei @ x

out.shape

In [ ]:
head_size = 16

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)


wei = (q @ k.transpose(-2, -1)) / 4
mask = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(mask == 0, float('-inf'))
wei = F.softmax(wei, -1)
wei[0]

out = wei @ value(x)

out.shape, wei[0]